In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [2]:
!python --version

Python 3.13.1


In [3]:
# Load the 2 data file into a Pandas DataFrame
# df1 = pd.read_csv("S_PatientsWithLabels-sep-tab.txt", sep="\t")
# df2 = pd.read_csv("S_PatientsWithFeatures_4logs_sep-tab.txt", sep="\t")
# Fusionner les deux tables en utilisant l'identifiant
# df = pd.merge(df2, df1, on='Patient_ID')  # df1 contient 'Target', df2 contient les variables
# df.to_csv('pour_lstm.csv', index=False)

In [4]:
df = pd.read_csv("pour_lstm.csv", sep=";")

In [5]:
# Regroupement par identifiant
grouped = df.groupby('Patient_ID')

# Créer listes de séquences et cibles
sequences = []
targets = []

for patient_id, group in grouped:
    target = group['Target'].iloc[0]
    # Sélectionner les 34 colonnes variables (ajustez si noms spécifiques)
    seq = group.iloc[:, 0:34].values  # colonnes 0 à 33
    sequences.append(torch.tensor(seq, dtype=torch.float))
    targets.append(target)

# Encoder les cibles si ce sont des classes non numériques
le = LabelEncoder()
targets_encoded = le.fit_transform(targets)
targets_tensor = torch.tensor(targets_encoded, dtype=torch.long)

In [6]:
# train , test
train_seqs, test_seqs, train_targets, test_targets = train_test_split(
    sequences, targets_tensor, test_size=0.1, random_state=42
)

In [7]:
# create dataset and loader
class SequenceDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

train_dataset = SequenceDataset(train_seqs, train_targets)
test_dataset = SequenceDataset(test_seqs, test_targets)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [25]:
#Définir le modèle
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # dernière sortie de la séquence
        out = self.fc(out)
        return out

model = LSTMClassifier(input_size=34, hidden_size=16, num_layers=2, num_classes=len(le.classes_))

In [26]:
# train the model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 1000

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_seqs, batch_targets in train_loader:
        outputs = model(batch_seqs)
        loss = criterion(outputs, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"\rEpoch {epoch+1}/{num_epochs}, Perte moyenne: {total_loss/len(train_loader):.4f}", end='', flush=True)

Epoch 1000/1000, Perte moyenne: 0.3213

In [27]:
def evaluate(model, dataloader):
    model.eval()
    all_targets = []
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for batch_seqs, batch_targets in dataloader:
            outputs = model(batch_seqs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_targets.extend(batch_targets.numpy())
            if probs.shape[1] == 2:
                all_probs.extend(probs[:,1].numpy())  # probabilité classe 1
            else:
                all_probs.extend(probs.numpy()[:,1])  # pour multiclasses
            all_preds.extend(preds.numpy())

    accuracy = accuracy_score(all_targets, all_preds)
    auc = roc_auc_score(all_targets, all_probs)
    return accuracy, auc

In [28]:
train_acc, train_auc = evaluate(model, train_loader)
test_acc, test_auc = evaluate(model, test_loader)

print(f"Train Accuracy: {train_acc:.3f}, Train AUC: {train_auc:.3f}")
print(f"Test Accuracy: {test_acc:.3f}, Test AUC: {test_auc:.3f}")

rob=test_auc/train_auc
print(rob)

Train Accuracy: 0.891, Train AUC: 0.956
Test Accuracy: 0.833, Test AUC: 0.900
0.9411858755754885
